In [20]:
import os, sys, gc, time, json, re
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import faiss

# Paths
SOURCE1_PATH = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\preprocessed_train_sounce1.tsv')
SOURCE2_PATH = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\preprocessed_train_sounce2.tsv')
SOURCE3_PATH = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\preprocessed_train_sounce3.tsv')

CANDIDATE_IDS_PATH = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\model_output\candidate_ids.npy')
FAISS_INDEX_PATH = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\model_output\candidate_index.faiss')
MODEL_OUTPUT = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\src\checkpoints')

# FORCED CPU INFERENCE
# DirectML (Intel GPU) often has silent calculation bugs with CNN masked_fill/max pooling,
# which produces garbage embeddings (low scores ~0.4 instead of 0.8+).
DEVICE = torch.device('cpu')
print("Using CPU for accurate inference:", DEVICE)


Using CPU for accurate inference: cpu


In [21]:
COLS = ["entity_id", "name_norm", "name_ascii", "address_norm", "country_norm", "legal_form"]

print("1. Loading Source 1, 2 and 3 so we can print candidate names properly...")
df_s1 = pd.read_csv(SOURCE1_PATH, sep="\t", usecols=COLS, dtype=str).fillna("")
df_s2 = pd.read_csv(SOURCE2_PATH, sep="\t", usecols=COLS, dtype=str).fillna("")
df_s3 = pd.read_csv(SOURCE3_PATH, sep="\t", usecols=COLS, dtype=str).fillna("")

records = pd.concat([df_s1, df_s2, df_s3], ignore_index=True)
records = records.drop_duplicates("entity_id").set_index("entity_id", drop=False)
print(f"Total Unique Records for Name Lookup: {len(records):,}")

print("2. Loading Candidate IDs...")
candidate_ids = np.load(CANDIDATE_IDS_PATH, allow_pickle=True)
print("3. Loading FAISS Index...")
faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))
print("Loaded Data & Index Successfully!")


1. Loading Source 1, 2 and 3 so we can print candidate names properly...
Total Unique Records for Name Lookup: 12,527,040
2. Loading Candidate IDs...
3. Loading FAISS Index...
Loaded Data & Index Successfully!


In [22]:
# ==========================================
# Model Classes for Query Embedding Generation
# ==========================================
TEXT_FIELDS = ("name_norm", "name_ascii", "address_norm", "country_norm")

def split_legal_forms(value: str) -> list[str]:
    value = str(value or "").strip()
    if value.startswith("["):
        try:
            value = ",".join(json.loads(value))
        except (json.JSONDecodeError, TypeError):
            pass
    strip_chars = " '\""
    tokens = re.split(r"[,;|]", value)
    result = []
    for x in tokens:
        cleaned = x.strip(strip_chars)
        if cleaned and cleaned != "missing":
            result.append(cleaned)
    return result

class CharTokenizer:
    PAD, UNK = 0, 1
    def __init__(self, vocab: dict[str, int]):
        self.vocab = vocab
    def encode(self, values, length: int) -> torch.Tensor:
        output = np.zeros((len(values), length), dtype=np.int64)
        for row, value in enumerate(values):
            ids = [self.vocab.get(char, self.UNK) for char in str(value)[:length]]
            output[row, :len(ids)] = ids
        return torch.from_numpy(output)
    @classmethod
    def load(cls, path):
        return cls(json.loads(Path(path).read_text(encoding="utf-8")))

class LegalFormVocab:
    PAD, UNK = 0, 1
    def __init__(self, vocab: dict[str, int]):
        self.vocab = vocab
    def encode(self, values) -> torch.Tensor:
        forms = [split_legal_forms(value) for value in values]
        width = max(1, max(map(len, forms), default=0))
        output = np.zeros((len(forms), width), dtype=np.int64)
        for row, values_for_row in enumerate(forms):
            if values_for_row:
                output[row, :len(values_for_row)] = [self.vocab.get(value, self.UNK) for value in values_for_row]
        return torch.from_numpy(output)
    @classmethod
    def load(cls, path):
        return cls(json.loads(Path(path).read_text(encoding="utf-8")))

class CharCNN(nn.Module):
    def __init__(self, vocab_size: int, char_dim: int = 48, channels: int = 64, output_dim: int = 128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, char_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(char_dim, channels, kernel_size=kernel, padding=kernel // 2)
            for kernel in (3, 5, 7)
        ])
        self.projection = nn.Linear(channels * 3, output_dim)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        present = token_ids.ne(0)
        x = self.embedding(token_ids).transpose(1, 2)
        pooled = []
        for conv in self.convs:
            value = F.relu(conv(x)).masked_fill(~present.unsqueeze(1), -1e4)
            value = value.max(dim=-1).values
            pooled.append(torch.where(present.any(1, keepdim=True), value, torch.zeros_like(value)))
        return self.projection(torch.cat(pooled, dim=1))

class EntityEncoder(nn.Module):
    def __init__(self, char_vocab_size: int, legal_vocab_size: int,
                 char_dim: int = 48, channels: int = 64, field_dim: int = 128,
                 legal_dim: int = 16, output_dim: int = 128, dropout: float = 0.15):
        super().__init__()
        self.name_cnn = CharCNN(char_vocab_size, char_dim, channels, field_dim)
        self.address_cnn = CharCNN(char_vocab_size, char_dim, channels, field_dim)
        self.legal_embedding = nn.Embedding(legal_vocab_size, legal_dim, padding_idx=0)
        self.mlp = nn.Sequential(
            nn.Linear(field_dim * 4 + legal_dim, 256),
            nn.LayerNorm(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, output_dim),
        )

    def forward(self, batch: dict[str, torch.Tensor]) -> torch.Tensor:
        legal_ids = batch["legal_form"]
        legal_vectors = self.legal_embedding(legal_ids)
        legal_mask = legal_ids.ne(0).unsqueeze(-1)
        legal = (legal_vectors * legal_mask).sum(1) / legal_mask.sum(1).clamp_min(1)
        features = torch.cat([
            self.name_cnn(batch["name_norm"]),
            self.name_cnn(batch["name_ascii"]),
            self.address_cnn(batch["address_norm"]),
            self.address_cnn(batch["country_norm"]),
            legal,
        ], dim=1)
        return F.normalize(self.mlp(features), p=2, dim=1)

def records_to_batch(records: pd.DataFrame, tokenizer: CharTokenizer,
                     legal_vocab: LegalFormVocab, device, name_length: int = 112,
                     address_length: int = 224, country_length: int = 32):
    return {
        "name_norm": tokenizer.encode(records["name_norm"], name_length).to(device),
        "name_ascii": tokenizer.encode(records["name_ascii"], name_length).to(device),
        "address_norm": tokenizer.encode(records["address_norm"], address_length).to(device),
        "country_norm": tokenizer.encode(records["country_norm"], country_length).to(device),
        "legal_form": legal_vocab.encode(records["legal_form"]).to(device),
    }


In [23]:
# Load Model Weights
print(f"Loading weights from: {MODEL_OUTPUT}")
eval_tokenizer = CharTokenizer.load(MODEL_OUTPUT / "tokenizer.json")
eval_legal_vocab = LegalFormVocab.load(MODEL_OUTPUT / "legal_form_vocab.json")

eval_model = EntityEncoder(
    char_vocab_size=len(eval_tokenizer.vocab) + 2,
    legal_vocab_size=len(eval_legal_vocab.vocab) + 2
).to(DEVICE)

# Weights are loaded onto CPU initially since you're training on Kaggle GPU
checkpoint = torch.load(MODEL_OUTPUT / "best.pt", map_location='cpu')
eval_model.load_state_dict(checkpoint["model"])
eval_model.eval()

print("Model loaded successfully!")


Loading weights from: C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\src\checkpoints
Model loaded successfully!


C:\Users\ps302\AppData\Local\Temp\ipykernel_2528\1436694119.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(MODEL_OUTPUT / "best.pt", map_locati

In [24]:
# ==============================================================================
# Ultra-Fast FAISS Prediction Function
# ==============================================================================
def predict_matches_faiss(query_id: str, top_k: int = 10):
    if query_id not in records.index:
        print(f"[ERROR] Query ID '{query_id}' not found in records.")
        return None

    query_row = records.loc[[query_id]]
    eval_model.eval()
    
    with torch.no_grad():
        batch_inputs = records_to_batch(query_row, eval_tokenizer, eval_legal_vocab, DEVICE)
        query_emb = eval_model(batch_inputs).cpu().numpy().astype(np.float32)

    t0 = time.time()
    scores, indices = faiss_index.search(query_emb, top_k)
    search_time_ms = (time.time() - t0) * 1000

    scores = scores[0]
    indices = indices[0]

    q = query_row.iloc[0]
    print("\n" + "=" * 80)
    print(f"QUERY ID  : {query_id}")
    print(f"Name      : {q.get('name_norm', '')}")
    print(f"Address   : {q.get('address_norm', '')}")
    print(f"Search Time: {search_time_ms:.2f} ms")
    print("=" * 80)
    print(f"FAISS TOP {top_k} PREDICTED MATCHES:")
    print("=" * 80)

    for rank, (cand_idx, score) in enumerate(zip(indices, scores), start=1):
        cand_id = candidate_ids[cand_idx]
        print(f"Rank {rank} | Score: {score:.4f} | ID: {cand_id}")
        
        if cand_id in records.index:
            cand_row = records.loc[cand_id]
            print(f"  Name      : {cand_row.get('name_norm', '')}")
            print(f"  Address   : {cand_row.get('address_norm', '')}")
            print(f"  Country   : {cand_row.get('country_norm', '')}")
            print(f"  Legal Form: {cand_row.get('legal_form', '')}")
        else:
            print("  [Name Details Not Found in Source 1/2/3]")
        print("-" * 80)


In [25]:
print("Loading labeledPairs to find actual queries with known matches...")
LABELS_PATH = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\labeledPairs.tsv')
labels_df = pd.read_csv(LABELS_PATH, sep="\t", dtype=str)
labels_df["label"] = pd.to_numeric(labels_df["label"], errors="coerce").fillna(0).astype(int)

# Extract queries that actually have a positive match (label == 1)
valid_queries = labels_df[labels_df["label"] == 1]["source1_entity_id"].unique().tolist()

print(f"\nTotal Valid Queries in dataset: {len(valid_queries)}")
print("Testing the model on 5 GUARANTEED queries to show accurate recommendations:\n")

for q_id in valid_queries[:5]:
    predict_matches_faiss(q_id, top_k=5)


Loading labeledPairs to find actual queries with known matches...

Total Valid Queries in dataset: 2083574
Testing the model on 5 GUARANTEED queries to show accurate recommendations:


QUERY ID  : S1-925783039
Name      : orelees barbershop
Address   : 1795 westchester drive high point nc
Search Time: 11477.89 ms
FAISS TOP 5 PREDICTED MATCHES:
Rank 1 | Score: 0.4363 | ID: S2-306487984
  Name      : pitchf0rd kan and mcsweeney
  Address   : 607 oklahoma avenue sweetwater citty tx
  Country   : us
  Legal Form: missing
--------------------------------------------------------------------------------
Rank 2 | Score: 0.4341 | ID: S2-97451468
  Name      : hand freeman and pickett service
  Address   : 9452 sh 75 new waverly tx
  Country   : us
  Legal Form: incorporated
--------------------------------------------------------------------------------
Rank 3 | Score: 0.4306 | ID: S2-162098063
  Name      : hannah futurewave
  Address   : 8677 bonview terrace clarence ny
  Country   : us
  Leg